# Week 6 Workshop (Langauge Models) Solution
<author>&copy; Prepared by Professor Yuefeng Li (QUT) </author>

In [1]:
#Task 1: Function index_docs() returnes an index - a dictionary {term:{docID1:freq1, DocID2:freq2, ...}, …}

import glob, os
import string
from stemming.porter2 import stem

def index_docs(inputpath,stop_words):
    Index = {}    # initialize the index
    os.chdir(inputpath)
    for file_ in glob.glob("*.xml"):
        start_end = False
        for line in open(file_):
            line = line.strip()
            if(start_end == False):
                if line.startswith("<newsitem "):
                    for part in line.split():
                        if part.startswith("itemid="):
                            docid = part.split("=")[1].split("\"")[1]
                            break  
                if line.startswith("<text>"):
                    start_end = True  
            elif line.startswith("</text>"):
                break
            else:
                line = line.replace("<p>", "").replace("</p>", "")
                line = line.translate(str.maketrans('','', string.digits)).translate(str.maketrans(string.punctuation, ' '*len(string.punctuation)))
                for term in line.split():
                    term = stem(term.lower())
                    if len(term) > 2 and term not in stop_words:
                        try:
                            try:
                                Index[term][docid] += 1
                            except KeyError:
                                Index[term][docid]=1
                        except KeyError:  
                            Index[term] = {docid:1} 
    return Index

ModuleNotFoundError: No module named 'stemming'

In [ ]:
# Task 2: define a function likelihood_IR(I,Q) to estimate P(Q|D) for all documents D in a folder indexed by I

def likelihood_IR(I, Q):  # index I is a Dirctionary of term:Directionary of (itemId:freq)
    L={}    # L is the selected inverted list
    R={}    # R is a directionary of docId:score
    D_len={} # D_len is a directionary of docId:length
    for list in I.items():
        for id in list[1].items(): 
            R[id[0]]=1       # get all document IDs and initialize as 1
            D_len[id[0]]=0.5 # initialize a small non-zero value as it will be used as denominator
        if (list[0] in Q):     # select inverted lists based on the query
                L[list[0]]= I[list[0]]
    for q_term in Q.items(): # L may not include all query terms
        if not(q_term[0] in L):
            L[q_term[0]]={}
    
    for list in I.items():
        for id in list[1].items(): # Count term occurrences in documents
            D_len[id[0]]= D_len[id[0]] + id[1]  
    for (d, sd) in R.items():
        for (term, f) in L.items():
            if not(d in f):
                f[d]=0
            sd = sd*(f[d]/D_len[d]) # see the equation in wk5 lecture notes 
        R[d] = sd
    return R

In [ ]:
# Task 3: define Jelinek-Mercer smoothing function likelihood_JM(I_C, I_D, Q, lamda) to 
# estimate P(Q|D) for all documents D in a folder indexed by I

def likelihood_JM(I_C, I_D, Q, lamda):  # index I_C ans I_D have the form of {term:{itemId:freq}}
    # calcualte query term frequency in documents indexed by I_D
    L={}    # L is the selected inverted list
    R={}    # R is a directionary of docId:score
    D_len={} # D_len is a directionary of docId:length
    for list in I_D.items():
        for id in list[1].items(): 
            R[id[0]]=1       # get all document IDs and initialize as 1
            D_len[id[0]]=0.5 # initialize a small non-zero value as it will be used as denominator
        if (list[0] in Q):     # select inverted lists based on the query
                L[list[0]]= I_D[list[0]]
    for q_term in Q.items(): # L may not include all query terms
        if not(q_term[0] in L):
            L[q_term[0]]={}
    for list in I_D.items():
        for id in list[1].items(): # Count term occurrences in documents
            D_len[id[0]]= D_len[id[0]] + id[1]  
    # calculate query term frequency in I_C
    CF={}
    L_C={}
    for list in I_C.items():
        if (list[0] in Q):     # select inverted lists based on Q in I_C
                L_C[list[0]]= I_C[list[0]]
                CF[list[0]] = 0 # assign 0 to each query term
    for (term, doc) in L_C.items():
        for id in doc.items():
            CF[term] = CF[term] + id[1]
    C_len = 0
    for list in I_C.items():
        for id in list[1].items(): # Count term occurrences in documents
            C_len = C_len + id[1]  
    # using the equation        
    for (d, sd) in R.items():
        for (term, f) in L.items():
            if not(d in f):
                f[d]=0
            sd = sd*(((1-lamda)*f[d]/D_len[d])+(lamda*CF[term]/C_len)) # see page 12 in wk5 lecture notes
        R[d] = sd
    return R

In [ ]:
# Task 4: Test the defined functions

curr_path=os.getcwd()
stopwords_f = open('common-english-words.txt', 'r')
stop_words = stopwords_f.read().split(',')
stopwords_f.close()

C_path = curr_path+'/collection_C'
Index_C = index_docs(C_path, stop_words) #Index I_C for all terms in <text> in collection_C
os.chdir(curr_path)

Test_path = curr_path+'/Test_docs'
Index_Test = index_docs(Test_path, stop_words) #Index I_D for all terms in <text> in Test_docs 
os.chdir(curr_path)

Query1 = {'formula':1, 'one':1} 
Query2 = {'share':1}
Query3 = {'leaderboard':1, 'british':1}

#test the likelyhood IR model
result1 = likelihood_IR(Index_Test, Query2)
x1 = sorted(result1.items(), key=lambda x: x[1],reverse=True)
print('Likelihood Query Result ---------')
for (id, w) in x1:
    if w>0:
        print('Document ID: '+id + ' and relevance weight: ' + str(w))
            
# test the smoothing-based IR model
result2 = likelihood_JM(Index_C, Index_Test, Query2, 0.4)
x2 = sorted(result2.items(), key=lambda x: x[1],reverse=True)
print('JM Smoothing-Based Query Result ---------')
for (id, w) in x2:
    if w>0:
        print('Document ID: '+id + ' and relevance weight: ' + str(w))